<a href="https://colab.research.google.com/github/prrmzz/FashionMNIST-Parham/blob/main/Copy_of_FashionMNIST_Parham.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q tensorflow tensorflow-datasets

In [4]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

In [5]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available: 1


In [6]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.expand_dims(image, -1)
    return image, label

In [7]:
(ds_train, ds_test), ds_info = tfds.load(
    'fashion_mnist',
    split=['train', 'test'],
    as_supervised=True,
    with_info=True
)

In [8]:
batch_size = 64
ds_train = ds_train.map(preprocess).shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.map(preprocess).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [9]:
class PreActBlock(tf.keras.Model):
    def __init__(self, filters, strides=1):
        super(PreActBlock, self).__init__()
        self.bn1 = layers.BatchNormalization()
        self.relu1 = layers.ReLU()
        self.conv1 = layers.Conv2D(filters, 3, strides=strides, padding='same')
        self.bn2 = layers.BatchNormalization()
        self.relu2 = layers.ReLU()
        self.conv2 = layers.Conv2D(filters, 3, strides=1, padding='same')
        if strides != 1:
            self.shortcut = layers.Conv2D(filters, 1, strides=strides)
        else:
            self.shortcut = lambda x: x

    def call(self, x, training=False):
        shortcut = self.shortcut(x)
        x = self.bn1(x, training=training)
        x = self.relu1(x)
        x = self.conv1(x)
        x = self.bn2(x, training=training)
        x = self.relu2(x)
        x = self.conv2(x)
        return x + shortcut

def build_preact_resnet18(input_shape=(28, 28, 1), num_classes=10):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(64, 3, strides=1, padding='same')(inputs)
    x = PreActBlock(64)(x)
    x = PreActBlock(64)(x)
    x = PreActBlock(128, strides=2)(x)
    x = PreActBlock(128)(x)
    x = PreActBlock(256, strides=2)(x)
    x = PreActBlock(256)(x)
    x = PreActBlock(512, strides=2)(x)
    x = PreActBlock(512)(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    return model

model = build_preact_resnet18()

In [10]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(ds_train, epochs=10, validation_data=ds_test)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 107s 84ms/step - accuracy: 0.6132 - loss: 2.1036 - val_accuracy: 0.8103 - val_loss: 0.6022
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 120s 71ms/step - accuracy: 0.8213 - loss: 0.5388 - val_accuracy: 0.8435 - val_loss: 0.4341
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 71ms/step - accuracy: 0.8392 - loss: 0.5476 - val_accuracy: 0.8457 - val_loss: 0.4479
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 72ms/step - accuracy: 0.8324 - loss: 0.6259 - val_accuracy: 0.8783 - val_loss: 0.3461
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 71ms/step - accuracy: 0.8662 - loss: 0.4354 - val_accuracy: 0.8775 - val_loss: 0.3356
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 70s 74ms/step - accuracy: 0.9013 - loss: 0.2794 - val_accuracy: 0.8778 - val_loss: 0.3366
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 74ms/step - accuracy: 0.9065 - loss: 0.2581 - val_accuracy: 0.8959 - val_loss: 0.2851
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 75ms/step - accuracy: 0.9109 - loss: 0.2465 

In [11]:
test_loss, test_accuracy = model.evaluate(ds_test)
print(f'Test accuracy: {test_accuracy:.4f}')

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.8950 - loss: 0.2903
Test accuracy: 0.8943
